In [0]:
from pyspark.sql.functions import *

SILVER_TABLE = "workspace.healthcare_demo.silver_patient_events"

GOLD_PATIENT_SUMMARY = "workspace.healthcare_demo.gold_patient_summary"
GOLD_WARD_SUMMARY = "workspace.healthcare_demo.gold_ward_summary"
GOLD_DEVICE_SUMMARY = "workspace.healthcare_demo.gold_device_summary"

In [0]:
silver_df = (
    spark.read
         .table(SILVER_TABLE)
)

In [0]:
display(silver_df)

In [0]:
patient_summary = (
    silver_df
        .groupBy("patient_id")
        .agg(
            avg("heart_rate").alias("avg_heart_rate"),
            avg("spo2").alias("avg_spo2"),
            avg("temperature").alias("avg_temperature"),
            count("*").alias("total_events")
        )
)

In [0]:
(
    patient_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_PATIENT_SUMMARY)
)

In [0]:
display(spark.table(GOLD_PATIENT_SUMMARY))

In [0]:
ward_summary = (
    silver_df
        .groupBy("ward")
        .agg(
            avg("heart_rate").alias("avg_heart_rate"),
            avg("spo2").alias("avg_spo2"),
            avg("temperature").alias("avg_temperature"),
            countDistinct("patient_id").alias("patient_count"),
            count("*").alias("total_events")
        )
)

In [0]:
display(ward_summary)



In [0]:
(
    ward_summary.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_WARD_SUMMARY)
)

In [0]:
display(spark.table(GOLD_WARD_SUMMARY))

In [0]:
device_summary = (
    silver_df
        .groupBy("device_id")
        .agg(
            avg("battery_level").alias("avg_battery_level"),
            avg("signal_strength").alias("avg_signal_strength"),
            count("*").alias("usage_count"),
            first("device_type").alias("device_type")
        )
)

In [0]:
display(device_summary)

In [0]:
spark.table(SILVER_TABLE).count()

In [0]:
from pyspark.sql.functions import *

SILVER_TABLE = "workspace.healthcare_demo.silver_patient_events"

GOLD_TABLE = "workspace.healthcare_demo.gold_patient_summary"

GOLD_CHECKPOINT = "/Volumes/workspace/healthcare_demo/checkpoints/gold"

In [0]:
silver_stream = (
    spark.readStream
        .table(SILVER_TABLE)
)

In [0]:
gold_stream = (
    silver_stream
        .groupBy("hospital_id", "ward")
        .agg(
            count("*").alias("total_patients"),
            avg("heart_rate").alias("avg_heart_rate"),
            avg("temperature").alias("avg_temperature"),
            avg("spo2").alias("avg_spo2"),
            avg("battery_level").alias("avg_battery_level")
        )
)

In [0]:
gold_query = (
    gold_stream.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", GOLD_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(GOLD_TABLE)
)

In [0]:
%sql
DROP TABLE IF EXISTS workspace.healthcare_demo.gold_patient_summary;

In [0]:
dbutils.fs.rm(
    "/Volumes/workspace/healthcare_demo/checkpoints/gold",
    True
)

In [0]:
gold_stream = (
    silver_stream
        .groupBy("hospital_id", "ward")
        .agg(
            count("*").alias("total_patients"),
            avg("heart_rate").alias("avg_heart_rate"),
            avg("temperature").alias("avg_temperature"),
            avg("spo2").alias("avg_spo2"),
            avg("battery_level").alias("avg_battery_level")
        )
)

In [0]:
gold_query = (
    gold_stream.writeStream
        .format("delta")
        .outputMode("complete")
        .option(
            "checkpointLocation",
            "/Volumes/workspace/healthcare_demo/checkpoints/gold"
        )
        .trigger(availableNow=True)
        .toTable("workspace.healthcare_demo.gold_patient_summary")
)

In [0]:
%sql
SELECT *
FROM workspace.healthcare_demo.gold_patient_summary
ORDER BY hospital_id, ward;

In [0]:
from pyspark.sql.functions import *

In [0]:
silver_df = spark.table("workspace.healthcare_demo.silver_patient_events")

gold_hospital_summary = (
    silver_df
        .groupBy("hospital_id")
        .agg(
            count("*").alias("total_events"),
            countDistinct("patient_id").alias("unique_patients"),
            avg("heart_rate").alias("avg_heart_rate"),
            avg("spo2").alias("avg_spo2"),
            avg("temperature").alias("avg_temperature"),
            avg("battery_level").alias("avg_battery_level")
        )
)

gold_hospital_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.healthcare_demo.gold_hospital_summary")

In [0]:
gold_ward_summary = (
    silver_df
        .groupBy("hospital_id", "ward")
        .agg(
            count("*").alias("total_events"),
            countDistinct("patient_id").alias("unique_patients"),
            avg("heart_rate").alias("avg_heart_rate"),
            avg("spo2").alias("avg_spo2"),
            avg("temperature").alias("avg_temperature")
        )
)

gold_ward_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.healthcare_demo.gold_ward_summary")

In [0]:
gold_device_summary = (
    silver_df
        .groupBy("device_type")
        .agg(
            count("*").alias("total_events"),
            countDistinct("device_id").alias("unique_devices"),
            avg("battery_level").alias("avg_battery_level"),
            avg("signal_strength").alias("avg_signal_strength")
        )
)

gold_device_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.healthcare_demo.gold_device_summary")

In [0]:
gold_patient_alerts = (
    silver_df
        .filter(
            (col("heart_rate") > 120) |
            (col("heart_rate") < 50) |
            (col("spo2") < 92) |
            (col("temperature") > 38.5) |
            (col("systolic_bp") > 150) |
            (col("diastolic_bp") > 95)
        )
        .select(
            "event_id",
            "event_timestamp",
            "hospital_id",
            "ward",
            "room_number",
            "patient_id",
            "patient_age",
            "patient_gender",
            "heart_rate",
            "spo2",
            "temperature",
            "systolic_bp",
            "diastolic_bp",
            "device_id",
            "device_type"
        )
)

gold_patient_alerts.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.healthcare_demo.gold_patient_alerts")

In [0]:
%sql
SHOW TABLES IN workspace.healthcare_demo;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.healthcare_demo.gold_patient_alerts;

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.healthcare_demo.gold_hospital_summary;

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.healthcare_demo.gold_ward_summary;

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.healthcare_demo.gold_device_summary;

In [0]:
%sql
SELECT *
FROM workspace.healthcare_demo.gold_hospital_summary
ORDER BY total_events DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.healthcare_demo.gold_hospital_summary_clean AS
SELECT *
FROM workspace.healthcare_demo.gold_hospital_summary
WHERE hospital_id NOT IN ('H001', 'H002');

In [0]:
%sql
SELECT *
FROM workspace.healthcare_demo.gold_hospital_summary_clean
ORDER BY total_events DESC;

In [0]:
%sql
OPTIMIZE workspace.healthcare_demo.bronze_patient_events;

In [0]:
%sql
OPTIMIZE workspace.healthcare_demo.silver_patient_events;

In [0]:
%sql
OPTIMIZE workspace.healthcare_demo.gold_patient_alerts;

In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.healthcare_demo.bronze_patient_events) AS bronze_count,
    (SELECT COUNT(*) FROM workspace.healthcare_demo.silver_patient_events) AS silver_count,
    (SELECT COUNT(*) FROM workspace.healthcare_demo.gold_patient_alerts) AS alert_count;

In [0]:
%sql
OPTIMIZE workspace.healthcare_demo.gold_patient_alerts
ZORDER BY (hospital_id, patient_id);

In [0]:
%sql
SELECT COUNT(*) AS alert_count
FROM workspace.healthcare_demo.gold_patient_alerts;